In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import * 

In [2]:
spark = SparkSession.builder.appName("SparkOptimization").getOrCreate()

In [3]:
df = spark.range(1, 5000000)
sales_df = df.withColumn("CustomerID", floor(rand()*100000)) \
    .withColumn("RegionId",  floor(rand()*10)) \
    .withColumn("Amount", floor(rand()*1000))  

In [4]:
sales_df.count()

4999999

In [5]:
# Data Shuffling   --> groupBy  | Join | distinct | Sort | repartition 
#  No Shuffling    -->   select  | filter

In [6]:
sales_df.groupBy("RegionId").sum("Amount").explain(True)

== Parsed Logical Plan ==
'Aggregate ['RegionId], ['RegionId, sum(Amount#9L) AS sum(Amount)#28L]
+- Project [id#0L, CustomerID#2L, RegionId#5L, FLOOR((rand(7126285009071613739) * cast(1000 as double))) AS Amount#9L]
   +- Project [id#0L, CustomerID#2L, FLOOR((rand(6228985837038845845) * cast(10 as double))) AS RegionId#5L]
      +- Project [id#0L, FLOOR((rand(-2769288093603223558) * cast(100000 as double))) AS CustomerID#2L]
         +- Range (1, 5000000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
RegionId: bigint, sum(Amount): bigint
Aggregate [RegionId#5L], [RegionId#5L, sum(Amount#9L) AS sum(Amount)#28L]
+- Project [id#0L, CustomerID#2L, RegionId#5L, FLOOR((rand(7126285009071613739) * cast(1000 as double))) AS Amount#9L]
   +- Project [id#0L, CustomerID#2L, FLOOR((rand(6228985837038845845) * cast(10 as double))) AS RegionId#5L]
      +- Project [id#0L, FLOOR((rand(-2769288093603223558) * cast(100000 as double))) AS CustomerID#2L]
         +- Range (1, 5000000, step=1, spli

In [7]:
customer_df = sales_df.select("CustomerId").distinct() 

In [8]:
joined = sales_df.join(customer_df,"CustomerId")

In [9]:
joined.count()

4999999

In [10]:
joined.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [CustomerId])
:- Project [id#0L, CustomerID#2L, RegionId#5L, FLOOR((rand(7126285009071613739) * cast(1000 as double))) AS Amount#9L]
:  +- Project [id#0L, CustomerID#2L, FLOOR((rand(6228985837038845845) * cast(10 as double))) AS RegionId#5L]
:     +- Project [id#0L, FLOOR((rand(-2769288093603223558) * cast(100000 as double))) AS CustomerID#2L]
:        +- Range (1, 5000000, step=1, splits=Some(8))
+- Deduplicate [CustomerId#35L]
   +- Project [CustomerId#35L]
      +- Project [id#34L, CustomerID#35L, RegionId#5L, FLOOR((rand(7126285009071613739) * cast(1000 as double))) AS Amount#9L]
         +- Project [id#34L, CustomerID#35L, FLOOR((rand(6228985837038845845) * cast(10 as double))) AS RegionId#5L]
            +- Project [id#34L, FLOOR((rand(-2769288093603223558) * cast(100000 as double))) AS CustomerID#35L]
               +- Range (1, 5000000, step=1, splits=Some(8))

== Analyzed Logical Plan ==
CustomerID: bigint, id: bigint, RegionId:

In [11]:
# Broadcast Join 

In [12]:
from pyspark.sql.functions import broadcast

In [13]:
joined = sales_df.join(broadcast(customer_df),"CustomerId")

In [14]:
joined.count()

4999999

In [15]:
joined.explain(True)

== Parsed Logical Plan ==
'Join UsingJoin(Inner, [CustomerId])
:- Project [id#0L, CustomerID#2L, RegionId#5L, FLOOR((rand(7126285009071613739) * cast(1000 as double))) AS Amount#9L]
:  +- Project [id#0L, CustomerID#2L, FLOOR((rand(6228985837038845845) * cast(10 as double))) AS RegionId#5L]
:     +- Project [id#0L, FLOOR((rand(-2769288093603223558) * cast(100000 as double))) AS CustomerID#2L]
:        +- Range (1, 5000000, step=1, splits=Some(8))
+- ResolvedHint (strategy=broadcast)
   +- Deduplicate [CustomerId#50L]
      +- Project [CustomerId#50L]
         +- Project [id#49L, CustomerID#50L, RegionId#5L, FLOOR((rand(7126285009071613739) * cast(1000 as double))) AS Amount#9L]
            +- Project [id#49L, CustomerID#50L, FLOOR((rand(6228985837038845845) * cast(10 as double))) AS RegionId#5L]
               +- Project [id#49L, FLOOR((rand(-2769288093603223558) * cast(100000 as double))) AS CustomerID#50L]
                  +- Range (1, 5000000, step=1, splits=Some(8))

== Analyzed Lo

In [16]:
# 10-100 MB  of lookup -->  Use Broadcast Join 

In [17]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold",50*1024*1024)  # 50 MB 

In [19]:
spark.conf.get("spark.sql.shuffle.partitions")

'200'

In [20]:
spark.conf.set("spark.sql.shuffle.partitions",8)

In [21]:
spark.conf.get("spark.sql.shuffle.partitions")

'8'

In [22]:
from pyspark.sql.functions import when

In [23]:
skewed_df = sales_df.withColumn("RegionId",  when(  rand()<0.8, 1 ).otherwise(floor(rand()*10))   )

In [25]:
skewed_df.groupBy("RegionId").count().show()

+--------+-------+
|RegionId|  count|
+--------+-------+
|       2| 100533|
|       9| 100583|
|       7| 100580|
|       3|  99634|
|       5|  99619|
|       8|  99883|
|       4| 100198|
|       1|4099880|
|       0|  99558|
|       6|  99531|
+--------+-------+



In [26]:
# Salting technique

In [27]:
salted_df =  skewed_df.withColumn("salt",floor(rand()*5)).withColumn("region_salted", concat("RegionId", lit("_"), "salt"))

In [28]:
salted_df

DataFrame[id: bigint, CustomerID: bigint, RegionId: bigint, Amount: bigint, salt: bigint, region_salted: string]

In [30]:
salted_df.groupBy("region_salted").count().orderBy("region_salted").show()

+-------------+------+
|region_salted| count|
+-------------+------+
|          0_0| 19842|
|          0_1| 19883|
|          0_2| 20089|
|          0_3| 19767|
|          0_4| 19977|
|          1_0|818978|
|          1_1|820974|
|          1_2|820950|
|          1_3|820034|
|          1_4|818944|
|          2_0| 19969|
|          2_1| 20064|
|          2_2| 20121|
|          2_3| 20174|
|          2_4| 20205|
|          3_0| 20024|
|          3_1| 20090|
|          3_2| 19765|
|          3_3| 19986|
|          3_4| 19769|
+-------------+------+
only showing top 20 rows



In [32]:
# Caching 
# Not speed | All about reusing the computed result 

In [33]:
spark

In [39]:
sales_df.is_cached

True

In [43]:
sales_df.unpersist()

DataFrame[id: bigint, CustomerID: bigint, RegionId: bigint, Amount: bigint]

In [45]:
sales_df.is_cached

False

In [46]:
sales_df.groupBy("RegionId").sum("amount").show()
sales_df.groupBy("RegionId").sum("amount").show()

+--------+-----------+
|RegionId|sum(amount)|
+--------+-----------+
|       2|  250002820|
|       9|  249804773|
|       3|  248895180|
|       7|  249442545|
|       5|  249847470|
|       8|  250012946|
|       4|  248899430|
|       0|  250145978|
|       6|  249568943|
|       1|  249860061|
+--------+-----------+

+--------+-----------+
|RegionId|sum(amount)|
+--------+-----------+
|       2|  250002820|
|       9|  249804773|
|       3|  248895180|
|       7|  249442545|
|       5|  249847470|
|       8|  250012946|
|       4|  248899430|
|       0|  250145978|
|       6|  249568943|
|       1|  249860061|
+--------+-----------+



In [47]:
sales_df.cache()

DataFrame[id: bigint, CustomerID: bigint, RegionId: bigint, Amount: bigint]

In [48]:
sales_df.is_cached

True

In [50]:
sales_df.count()

4999999

In [51]:
sales_df.groupBy("RegionId").sum("amount").show()
sales_df.groupBy("RegionId").sum("amount").show()

+--------+-----------+
|RegionId|sum(amount)|
+--------+-----------+
|       2|  250002820|
|       9|  249804773|
|       3|  248895180|
|       7|  249442545|
|       5|  249847470|
|       8|  250012946|
|       4|  248899430|
|       0|  250145978|
|       6|  249568943|
|       1|  249860061|
+--------+-----------+

+--------+-----------+
|RegionId|sum(amount)|
+--------+-----------+
|       2|  250002820|
|       9|  249804773|
|       3|  248895180|
|       7|  249442545|
|       5|  249847470|
|       8|  250012946|
|       4|  248899430|
|       0|  250145978|
|       6|  249568943|
|       1|  249860061|
+--------+-----------+



In [52]:
# cache vs persist  

In [53]:
from pyspark import StorageLevel

In [54]:
sales_df.persist(StorageLevel.MEMORY_ONLY)

DataFrame[id: bigint, CustomerID: bigint, RegionId: bigint, Amount: bigint]

In [55]:
sales_df.persist(StorageLevel.MEMORY_AND_DISK)

DataFrame[id: bigint, CustomerID: bigint, RegionId: bigint, Amount: bigint]

In [56]:
sales_df.persist(StorageLevel.DISK_ONLY)

DataFrame[id: bigint, CustomerID: bigint, RegionId: bigint, Amount: bigint]

In [57]:
# Window Functions 

In [61]:
sales_data = [(1,"2025-01-01",101,1001,2,1200),
              (2,"2025-01-02",102,1002,1,800),
              (3,"2025-01-03",103,1003,4,2000),
              (4,"2025-01-04",103,1003,3,1500),
              (5,"2025-01-05",105,1005,2,1800),
              (6,"2025-01-06",103,1001,1,2200),
              (7,"2025-01-07",102,1003,1,3000) 
            ]
sales_df= spark.createDataFrame(sales_data, 
                                ["OrderID", "OrderDate", "CustomerID", "ProductID", "Quantity", "SalesAmount"])


In [62]:
sales_df.show()

+-------+----------+----------+---------+--------+-----------+
|OrderID| OrderDate|CustomerID|ProductID|Quantity|SalesAmount|
+-------+----------+----------+---------+--------+-----------+
|      1|2025-01-01|       101|     1001|       2|       1200|
|      2|2025-01-02|       102|     1002|       1|        800|
|      3|2025-01-03|       103|     1003|       4|       2000|
|      4|2025-01-04|       103|     1003|       3|       1500|
|      5|2025-01-05|       105|     1005|       2|       1800|
|      6|2025-01-06|       103|     1001|       1|       2200|
|      7|2025-01-07|       102|     1003|       1|       3000|
+-------+----------+----------+---------+--------+-----------+



In [66]:
del sales_df

In [78]:
data = [(1, "2025-01-01", 100),
        (1, "2025-01-02", 200),
        (1, "2025-01-03", 150), 
        (2, "2025-01-01", 300),
        (2, "2025-01-02", 250)
        ]  
order_df= spark.createDataFrame(data, 
                                [ "CustomerID", "OrderDate", "SalesAmount"])

In [79]:
order_df.show()

+----------+----------+-----------+
|CustomerID| OrderDate|SalesAmount|
+----------+----------+-----------+
|         1|2025-01-01|        100|
|         1|2025-01-02|        200|
|         1|2025-01-03|        150|
|         2|2025-01-01|        300|
|         2|2025-01-02|        250|
+----------+----------+-----------+



In [81]:
order_df.coalesce(1).write.format("csv").option("header","true").mode("overwrite").save("Files/Orders/")

In [83]:
order_df

DataFrame[CustomerID: bigint, OrderDate: string, SalesAmount: bigint]

In [84]:
order_df = order_df.withColumn("OrderDate", to_date("OrderDate"))

In [85]:
order_df

DataFrame[CustomerID: bigint, OrderDate: date, SalesAmount: bigint]

In [82]:
from pyspark.sql.window import Window

In [86]:
window_spec =  Window.partitionBy("CustomerID").orderBy("OrderDate")

In [87]:
order_df_running_total = order_df.withColumn("running_total", sum("SalesAmount").over(window_spec))

In [88]:
order_df_running_total.show()

+----------+----------+-----------+-------------+
|CustomerID| OrderDate|SalesAmount|running_total|
+----------+----------+-----------+-------------+
|         1|2025-01-01|        100|          100|
|         1|2025-01-02|        200|          300|
|         1|2025-01-03|        150|          450|
|         2|2025-01-01|        300|          300|
|         2|2025-01-02|        250|          550|
+----------+----------+-----------+-------------+



In [89]:
order_df_running_total.explain(True)

== Parsed Logical Plan ==
'Project [CustomerID#1566L, OrderDate#1591, SalesAmount#1568L, sum('SalesAmount) windowspecdefinition('CustomerID, 'OrderDate ASC NULLS FIRST, unspecifiedframe$()) AS running_total#1596]
+- Project [CustomerID#1566L, to_date(OrderDate#1567, None, Some(Asia/Calcutta), false) AS OrderDate#1591, SalesAmount#1568L]
   +- LogicalRDD [CustomerID#1566L, OrderDate#1567, SalesAmount#1568L], false

== Analyzed Logical Plan ==
CustomerID: bigint, OrderDate: date, SalesAmount: bigint, running_total: bigint
Project [CustomerID#1566L, OrderDate#1591, SalesAmount#1568L, running_total#1596L]
+- Project [CustomerID#1566L, OrderDate#1591, SalesAmount#1568L, running_total#1596L, running_total#1596L]
   +- Window [sum(SalesAmount#1568L) windowspecdefinition(CustomerID#1566L, OrderDate#1591 ASC NULLS FIRST, specifiedwindowframe(RangeFrame, unboundedpreceding$(), currentrow$())) AS running_total#1596L], [CustomerID#1566L], [OrderDate#1591 ASC NULLS FIRST]
      +- Project [Customer